# Defining global variables

In [1]:
REPO_NAME = 'Textual_Analysis_in_Finance'
BASE_DIR = f'/kaggle/working/{REPO_NAME}'
WEEK = 4

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Clone the lecture's git repo

In [3]:
!git clone https://github.com/minhtriphan/{REPO_NAME}.git
%cd {REPO_NAME}

Cloning into 'Textual_Analysis_in_Finance'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 168 (delta 43), reused 143 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 6.41 MiB | 46.24 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/kaggle/working/Textual_Analysis_in_Finance


# Let's have a look at FinBERT

FinBERT is a BERT-based language model specialized for financial documents (earnings call transcripts).

The model card is [**here**](https://huggingface.co/yiyanghkust/finbert-tone), where you can download the model using HuggingFace API.

Let's first try to download the model's configuration, tokenizer, and weights. This can be done via the `transformers` package. 

First, import the `AutoTokenizer`, `AutoConfig`, and `AutoModelForSequenceClassification`. They are the classes to specify the tokenizer, model configuration, and model architecture.

Second, specify the model we're gonna use as the `BACKBONE`.

In [4]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification

BACKBONE = 'yiyanghkust/finbert-tone'

## The tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
tokenizer

config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='yiyanghkust/finbert-tone', vocab_size=30873, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

You can also get the vocabulary size using `len(tokenizer)`

#### <span style="color:red">Question</span>

1. Can you explain the information in the tokenizer? Specifically, what are vocab_size, model_max_length, padding_side, truncation_side, etc.?

#### Let's tokenize some documents

In [6]:
sentences = [
    'The market volatility is high',
    'TSLA is underperformming the market'
]

tokenizer.tokenize(sentences[1])

['[UNK]', 'is', 'underperform', '##ming', 'the', 'market']

#### <span style="color:red">Question</span>

2. What happens to "underperformming"?
3. You can try tokenizing the following words:
   * `macroeconomics`
   * `receivables` and `recievables`
   * `overvaluation`
   * `hyperinflation`
   * `collateralization`
   * `decentralization` and `decentralized`

#### Encode tokens

After tokenizing the sentences, let's **convert tokens into numbers**.

We can do that by using `tokenizer.encode()`. The documentation is [**here**](https://huggingface.co/docs/transformers/en/main_classes/tokenizer#transformers.PythonBackend.encode). Important arguments include:

* `padding`: Activates and controls padding. You can set it to `True`, `'longest'`, or `'max_length'`. The default value is `False`, meaning do not pad
* `truncation`: Opposite to padding, this argument controls for truncation of long sentences. Set it to `True` (truncate to the `max_length`). The default value is `False`, meaning do not truncate
* `max_length`: The designed maximum length of all sentences
* `return_tensors`: Type of encoded texts. Default will return lists

#### Let's try it first without any of those arguments

In [7]:
encoded_text = tokenizer.encode(
    sentences[1]
)
encoded_text

[3, 2, 17, 1461, 6690, 6, 52, 4]

* You can reverse the process, meaning decode the encoded sentence

In [8]:
tokenizer.decode(encoded_text)

'[CLS] [UNK] is underperformming the market [SEP]'

#### Padding

* Try to pad (the sentence) to the maximum length, i.e., `padding = 'max_length'`

In [9]:
encoded_text = tokenizer.encode(
    sentences[1],
    padding = 'max_length',
    max_length = 10
)
encoded_text

[3, 2, 17, 1461, 6690, 6, 52, 4, 0, 0]

In [10]:
tokenizer.decode(encoded_text)

'[CLS] [UNK] is underperformming the market [SEP] [PAD] [PAD]'

* Try `padding = True` (same as `padding = 'longest'`)

With this choice, the tokenizer will pad the sentences to align with the longest sentence, **regardless** of the value of `max_length`. Therefore, to see the effect, we need to input more than one sentence.

In [11]:
encoded_text = tokenizer.encode(
    sentences,
    padding = True,
    max_length = 10
)
encoded_text

[[3, 2, 52, 1033, 17, 307, 4, 0], [3, 2, 17, 1461, 6690, 6, 52, 4]]

In [12]:
tokenizer.decode(encoded_text)

['[CLS] [UNK] market volatility is high [SEP] [PAD]',
 '[CLS] [UNK] is underperformming the market [SEP]']

#### Truncation

In [13]:
encoded_text = tokenizer.encode(
    sentences[1],
    padding = 'max_length',
    max_length = 5,
    truncation = True
)
encoded_text

[3, 2, 17, 1461, 4]

In [14]:
tokenizer.decode(encoded_text)

'[CLS] [UNK] is underperform [SEP]'

## The model configuration

The model configuration ... all aspects of model characteristics, ranging from the architecture, the dropout probabilities, hidden dimensions, activation functions, vocabulary size, maximum positional embedding size, etc.

You can find the definitions of all below values [**here**](https://huggingface.co/docs/transformers/en/model_doc/bert#transformers.BertConfig)

In [15]:
config = AutoConfig.from_pretrained(BACKBONE)
config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "Neutral",
    "1": "Positive",
    "2": "Negative"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "Negative": 2,
    "Neutral": 0,
    "Positive": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30873
}

Important values include:

* `vocab_size`: The vocabulary size, must be the same as the value in the tokenizer
* `max_position_embeddings`: This is the true limit of the model. The LM will not take texts longer than `max_position_embeddings` tokens
* `id2label`: A mapping between the encoded labels to sentiment labels. Only FinBERT has this value as it's a sentiment-analysis language model
* `label2id`: A reversed mapping of `id2label`

## The model

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(BACKBONE)
model

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30873, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

#### Explanation

There are some objects like `Embedding` and `Linear` together with others.

* `Embedding`: A class that converts encoded tokens into vectors
* `Linear`: A class that does linear projection. For example: `Linear(in_features = 768, out_features = 3)` will turn a 512 $\times$ 768 matrix, $X$, to a 512 $\times$ 3 matrix by $X \times W$, where $W$ is a 768 $\times$ 3 matrix

#### <span style="color:red">Question</span>

2. Can you explain the dimensions of `word_embeddings` and `position_embeddings`? How do they relate to the values in the model configuration above?

#### Let's try to use the model

* First, try tokenize

In [17]:
encoded_text = tokenizer.encode(
    sentences,
    padding = True,
    max_length = 10,
    return_tensors = 'pt'
)
encoded_text

tensor([[   3,    2,   52, 1033,   17,  307,    4,    0],
        [   3,    2,   17, 1461, 6690,    6,   52,    4]])

* Use the model

In [18]:
import torch

with torch.no_grad():
    output = model(encoded_text)

output

SequenceClassifierOutput(loss=None, logits=tensor([[-0.9036, -1.1680,  2.8827],
        [-3.4492, -1.9978,  8.2626]]), hidden_states=None, attentions=None)

In [19]:
output.logits.softmax(dim = -1)

tensor([[2.1804e-02, 1.6739e-02, 9.6146e-01],
        [8.1968e-06, 3.4991e-05, 9.9996e-01]])

#### <span style="color:red">Question</span>

3. What does this result mean?

In [20]:
sentiment_prediction = output.logits.softmax(dim = -1).argmax(dim = -1)
sentiment_prediction = sentiment_prediction.numpy().tolist()

for sentence, value in zip(sentences, sentiment_prediction):
    print(f'Sentence: {sentence} - sentiment: {config.id2label[value]}')

Sentence: The market volatility is high - sentiment: Negative
Sentence: TSLA is underperformming the market - sentiment: Negative


#### Let's try to put something very long to the model

In [21]:
encoded_text = tokenizer.encode(
    sentences,
    padding = 'max_length',
    max_length = 513,
    return_tensors = 'pt'
)

with torch.no_grad():
    output = model(encoded_text)

RuntimeError: The expanded size of the tensor (513) must match the existing size (512) at non-singleton dimension 1.  Target sizes: [2, 513].  Tensor sizes: [1, 512]

#### <span style="color:red">Question</span>

4. Why does the error happen?

#### Ok, now let's put the correct inputs to the model

In [22]:
encoded_text = tokenizer.encode(
    sentences,
    padding = 'max_length',
    max_length = 512,
    return_tensors = 'pt'
)

with torch.no_grad():
    output = model(encoded_text)
output

SequenceClassifierOutput(loss=None, logits=tensor([[ 3.0466, -1.6673, -3.9996],
        [ 3.0652, -1.8251, -3.6977]]), hidden_states=None, attentions=None)

* Let's compute the sentiment probabilities as before

In [23]:
sentiment_prediction = output.logits.softmax(dim = -1).argmax(dim = -1)
sentiment_prediction = sentiment_prediction.numpy().tolist()

for sentence, value in zip(sentences, sentiment_prediction):
    print(f'Sentence: {sentence} - sentiment: {config.id2label[value]}')

Sentence: The market volatility is high - sentiment: Neutral
Sentence: TSLA is underperformming the market - sentiment: Neutral


#### <span style="color:red">Question</span>

4. Why do we have different results now?

#### Ok, let's solve that problem

We need to tell the model to pay attention to non-pad tokens. To obtain that information, use `tokenizer()`, not `tokenizer.encode()`

In [24]:
encoded_text = tokenizer(
    sentences,
    padding = 'max_length',
    max_length = 512,
    return_tensors = 'pt',
    return_attention_mask = True
)
encoded_text

{'input_ids': tensor([[ 3,  2, 52,  ...,  0,  0,  0],
        [ 3,  2, 17,  ...,  0,  0,  0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [25]:
with torch.no_grad():
    output = model(input_ids = encoded_text['input_ids'], attention_mask = encoded_text['attention_mask'])

sentiment_prediction = output.logits.softmax(dim = -1).argmax(dim = -1)
sentiment_prediction = sentiment_prediction.numpy().tolist()

for sentence, value in zip(sentences, sentiment_prediction):
    print(f'Sentence: {sentence} - sentiment: {config.id2label[value]}')

Sentence: The market volatility is high - sentiment: Negative
Sentence: TSLA is underperformming the market - sentiment: Negative


# Running an LM in GPUs

To use GPU, click on "Session options" tab on the right-hand side panel.

Then, in the Accelerator drop down, choose the GPU you want. In our case, choose **GPU T4 x2**, meaning we will have 2 GPUs T4. 

A warning will pop up saying that we are granted 30 hours of GPU usage per week, so avoid abusing it.

The session (or this notebook) will restart.

First, tokenize the texts as normal. The `return_tensors` argument must be set to `'pt'`, meaning _Pytorch tensor_.

In [26]:
encoded_text = tokenizer(
    sentences,
    padding = 'max_length',
    max_length = 512,
    return_tensors = 'pt',
    return_attention_mask = True
)
encoded_text

{'input_ids': tensor([[ 3,  2, 52,  ...,  0,  0,  0],
        [ 3,  2, 17,  ...,  0,  0,  0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

Second, we check if the GPU is available

In [27]:
import torch

# To check if GPU is available. If you switched the GPU on, this should return True
torch.cuda.is_available()

True

Move the model and the tokenized inputs to the GPU

In [28]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model.to(device)
input_ids = encoded_text['input_ids'].to(device)
attention_mask = encoded_text['attention_mask'].to(device)

Prediction

In [29]:
with torch.no_grad():
    output = model(input_ids = input_ids, attention_mask = attention_mask)

#### Check the time

With 1 GPU T4

In [30]:
%%timeit
with torch.no_grad():
    output = model(input_ids = input_ids, attention_mask = attention_mask)

52.1 ms ± 192 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


With CPU

In [31]:
device = torch.device('cpu')
model.to(device)
input_ids = encoded_text['input_ids'].to(device)
attention_mask = encoded_text['attention_mask'].to(device)

In [32]:
%%timeit
with torch.no_grad():
    output = model(input_ids = input_ids, attention_mask = attention_mask)

963 ms ± 34.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


**It's around 30 times slower!!!**